# Full Graph

In the previous notebook, we build an a workflow with clear explicit steps.   

In this notebook, we will convert the previous flows as tool and create a react agent 

## Imports

In [ ]:
import sys 
sys.path.insert(0,"../code")

In [ ]:
import utils_display
import prompts
import utils
from utils_import import *
from langchain_core.messages import HumanMessage, SystemMessage
import tools
import rich

In [ ]:
from pydantic import BaseModel, Field
from typing import Sequence, Optional, Literal
from langchain_core.messages import BaseMessage

from typing_extensions import Annotated, Sequence
from langchain_core.messages import filter_messages

import operator
from langgraph.types import Command
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, get_buffer_string
from langgraph.graph import StateGraph, START, END


## State

In [ ]:

class AgentState(BaseModel):
    """
    State for the research agent scoping workflow.
    """

    # Research brief generated from user conversation history
    research_brief: Optional[str] = None
    # messages exchanged in the conversation
    messages: Annotated[Sequence[BaseMessage], operator.add] = []
    notes:str = ""
    final_report: str = ""


class WorkerAgentState(BaseModel):
    research_brief: Optional[str] = None
    messages: Annotated[Sequence[BaseMessage], operator.add] = []




In [ ]:
class ClarifyWithUserOutput(BaseModel):
    """Schema for user clarification decision and questions."""
    
    need_clarification: bool = Field(
        description="Whether the user needs to be asked a clarifying question.",
    )
    question: str = Field(
        description="A question to ask the user to clarify the report scope",
    )
    verification: str = Field(
        description="Verify message that we will start research after the user has provided the necessary information.",
    )


class ResearchQuestionOutput(BaseModel):
    """Schema for structured research brief generation."""
    
    research_brief: str = Field(
        description="A research question that will be used to guide the research.",
    )

In [ ]:
## Scoping methods

In [ ]:
model = ChatOpenAI(
        model="gpt-4.1",
        base_url=OPENAI_BASE_URL,
        temperature=0.0,
        #max_tokens=512,
)

In [ ]:
def clarify_with_user(state: AgentState) -> Command[Literal["write_research_brief", "__end__"]]:
    """
    Determine if the user's request contains sufficient information to proceed with research, or if a clarifying question is needed.
    """

    
    # Set up structured output model
    structured_output_model = model.with_structured_output(ClarifyWithUserOutput)

    # Invoke the model with clarification instructions
    response = structured_output_model.invoke([
        HumanMessage(content=prompts.clarify_with_user_instructions.format(
            messages=get_buffer_string(messages=state.messages), 
            date=utils.get_today_str()
        ))
    ])
    
    # Route based on clarification need
    if response.need_clarification:
        return Command(
            goto=END, 
            update={"messages": [AIMessage(content=response.question)]}
        )
    else:
        return Command(
            goto="write_research_brief", 
            update={"messages": [AIMessage(content=response.verification)]}
        )

def write_research_brief(state: AgentState):
    """
    Transform the conversation history into a comprehensive research brief.
    """
    # Set up structured output model
    structured_output_model = model.with_structured_output(ResearchQuestionOutput)
    
    # Generate research brief from conversation history
    response = structured_output_model.invoke([
        HumanMessage(content=prompts.transform_messages_into_research_topic.format(
            messages=get_buffer_string(state.messages),
            date=utils.get_today_str()
        ))
    ])
    
    return {
        "research_brief": response.research_brief,
        "messages": [HumanMessage(content=f"{response.research_brief}.")]
    }



def research(state: AgentState):
    llm_tools = [tools.web_search_tool, tools.think_tool, tools.summarize_findings_tool]
    llm = ChatOpenAI(
        model="gpt-4o",
        base_url=OPENAI_BASE_URL,
        temperature=0.7
    )
    system_message = prompts.agent_system_prompt.format(date=utils.get_today_str())
    research_agent = create_agent(
        llm,                          # The LLM to use
        llm_tools,                        # List of tools
        system_prompt=system_message # System prompt
    )


    state_research = WorkerAgentState (research_brief=state.research_brief, messages= [HumanMessage(content= state.research_brief)] )

    state_research = research_agent.invoke(state_research)

    notes = utils.extract_research_notes(state_research['messages'])

    return {"notes": notes , "messages": [AIMessage(content=f"Worker done researching. Notes {notes}")]}



def extract_research_notes(messages:list):
    """
    Extract research notes from the conversation messages.
    Filters messages to include only those generated by tools or the AI.
    """

    raw_notes = [
            str(m.content) for m in filter_messages(
                messages, 
                include_types=["tool", "ai"]
            )
        ]
    return "\n".join(raw_notes)


def final_report_generation(state: AgentState):
    """
    Final report generation node.

    Synthesizes all research findings into a comprehensive final report
    """

    writer_model = ChatOpenAI(
        model="gpt-4.1",
        base_url=OPENAI_BASE_URL,
        max_tokens=32000
    ) 

    # Pull the aggregated research notes captured by previous steps
    notes = extract_research_notes(state.messages)


    final_report_prompt = prompts.final_report_generation.format(
        research_brief=state.research_brief,
        findings=notes,
        date=utils.get_today_str()
    )

    # Use the writer model to generate the final report
    final_report = writer_model.invoke([HumanMessage(content=final_report_prompt)])

    return {
        "final_report": final_report.content, 
         "messages": [AIMessage(content=f" Generated final report. {final_report.content}")]
    }


In [ ]:
graph_builder = StateGraph(AgentState)

# Add workflow nodes
graph_builder.add_node("clarify_with_user", clarify_with_user)
graph_builder.add_node("write_research_brief", write_research_brief)
graph_builder.add_node("research", research)
graph_builder.add_node("report", final_report_generation)

# add edges
graph_builder.add_edge(START, "clarify_with_user") 
graph_builder.add_edge("write_research_brief", "research") 
graph_builder.add_edge("research", "report") 
graph_builder.add_edge("report", END)


# Compile the full workflow
graph = graph_builder.compile()
graph

In [ ]:
query = "What is the best coffee shop in seattle."

In [ ]:
result = graph.invoke({"messages": [HumanMessage(content=query)]})


In [ ]:
utils_display.format_messages(result['messages'])

In [ ]:
human_answer = "Quality of the coffee bean and the atmosphere of the shop are the two most important factors for me."
result["messages"] += [HumanMessage(content=human_answer)]
result = graph.invoke(result)



In [ ]:
with tracer.start_as_current_span("tool_search"):
    result = graph.invoke(result)

In [ ]:
result

In [ ]:
print (result['final_report'])

In [ ]:
from rich.markdown import Markdown

Markdown(result['final_report'])

In [ ]:
utils_display.format_messages(result['messages'])

In [ ]:
rich.print (result)

In [ ]:
from IPython.display import Image, display

Image("../images/trace_full_graph.png")